In [7]:
import pandas as pd
import csv

PATH = '...\R3_BreadBasket-1.csv'
data = pd.read_csv(PATH)

data['Item'].value_counts()

Item
Coffee            5471
Bread             3325
Tea               1435
Cake              1025
Pastry             856
                  ... 
Chicken sand         1
The BART             1
Adjustment           1
Olum & polenta       1
Bacon                1
Name: count, Length: 95, dtype: int64

In [12]:
# Data cleansing and exploratory analysis

data_cleaned = data[data['Item'] != 'NONE'] 
grouped_data = data.groupby('Transaction')
longitud_agrupado_por_transacciones = len(grouped_data.size())
productos_por_transaccion = len(data) / longitud_agrupado_por_transacciones
productos_por_transaccion

2.2340782709054663

In [13]:
data['Item'].nunique()

95

In [15]:
data['Item'].value_counts()
transactions_per_product_sorted = data.groupby('Item')['Transaction'].nunique().sort_values(ascending=False)
transactions_per_product_sorted = transactions_per_product_sorted.drop('NONE', errors='ignore')
top_10_products = transactions_per_product_sorted.head(10) 
top_10_products

Item
Coffee           4528
Bread            3097
Tea              1350
Cake              983
Pastry            815
Sandwich          680
Medialuna         585
Hot chocolate     552
Cookies           515
Brownie           379
Name: Transaction, dtype: int64

In [18]:
# Market basket analysis

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

data['Single transaction'] = data['Transaction'].astype(str)+'_'+data['Date'].astype(str)+' '+data['Time'].astype(str)
data = data.drop( columns=['Date', 'Time', 'Transaction']) 
data_binary = pd.crosstab(data['Single transaction'], data['Item']) 

def encode(item_freq):
    return item_freq.apply(lambda x: 1 if x > 0 else 0)

basket_input = data_binary.apply(encode) 
frequent_itemsets = apriori(basket_input, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.2)

rules_filtered = rules[rules['lift'] >= 1]
rules = rules_filtered.sort_values(by='confidence', ascending=False)
rules = rules[~rules['antecedents'].apply(lambda x: 'NONE' in x)]
rules

D:\Documentos\Educación\ejercicios python\env\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:109: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
26,(Toast),(Coffee),0.033365,0.475081,0.023502,0.704403,1.482699,0.007651,1.775789,0.336791
24,(Spanish Brunch),(Coffee),0.018046,0.475081,0.010807,0.598837,1.260494,0.002233,1.308493,0.210458
17,(Medialuna),(Coffee),0.061379,0.475081,0.034939,0.569231,1.198175,0.005779,1.218561,0.176213
20,(Pastry),(Coffee),0.085510,0.475081,0.047214,0.552147,1.162216,0.006590,1.172079,0.152626
1,(Alfajores),(Coffee),0.036093,0.475081,0.019515,0.540698,1.138116,0.002368,1.142861,0.125899
16,(Juice),(Coffee),0.038296,0.475081,0.020460,0.534247,1.124537,0.002266,1.127031,0.115155
21,(Sandwich),(Coffee),0.071346,0.475081,0.037981,0.532353,1.120551,0.004086,1.122468,0.115847
12,(Cake),(Coffee),0.103137,0.475081,0.054349,0.526958,1.109196,0.005350,1.109667,0.109767
22,(Scone),(Coffee),0.034309,0.475081,0.017941,0.522936,1.100729,0.001642,1.100310,0.094762
14,(Cookies),(Coffee),0.054034,0.475081,0.028014,0.518447,1.091280,0.002343,1.090053,0.088422


In [ ]:
"""
Insights

With our analysis, we can make some recommendations:

1. Coffee is associated with many other products, so it makes sense to run promotions with these types of products. 
Obviously, the more trust and support there is, the better the promotion will be. For example, it might be profitable to run promotions for toast and 
coffee, Spanish brunch and coffee, or croissants and coffee.

2. I wouldn't limit myself to just offering deals with coffee, but also with tea and cake, for instance. We can explore different types of pairings to diversify the offerings.

3. Often, the consequent is coffee (or tea, to a lesser extent). Therefore, we could focus on the products that precede these two and, 
when someone orders them, directly offer tea or coffee. This helps improve the shopping experience.
"""
